# Phase 3 — Model Training, Evaluation, and Selection

This notebook covers the training, validation, threshold optimization, business cost modeling, probability calibration, SHAP explainability, and final test evaluation of the AI Fraud Risk Manager model.

## Notebook Structure:
1. **Baseline Model:** Logistic Regression with preprocessors.
2. **Primary & Secondary Models:** LightGBM and XGBoost (Standard and Class-Weighted variants).
3. **Model Comparison:** Evaluating models using Precision, Recall, F1, PR-AUC, and ROC-AUC.
4. **Threshold Optimization & Cost Analysis:** Selecting the operating threshold that minimizes expected financial cost.
5. **Calibration Analysis:** Assessing if predicted probabilities map to actual fraud frequencies.
6. **Explainability (SHAP & Importance):** Global feature importance and local SHAP transaction explanations.
7. **Untouched Test Evaluation:** Final single run on chronological test set to estimate generalization performance.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import gc
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

# Append project root src to path
sys.path.append(os.path.abspath('..'))
sns.set_theme(style="whitegrid")
print("Environment initialized.")

### 1. Load Data

We load the processed pickle data splits generated in Phase 2.

In [ ]:
data_dir = "../data/processed"

print("Loading features...")
train_df = pd.read_pickle(os.path.join(data_dir, "train_features.pkl"))
val_df = pd.read_pickle(os.path.join(data_dir, "val_features.pkl"))
test_df = pd.read_pickle(os.path.join(data_dir, "test_features.pkl"))

drop_cols = ['isFraud', 'TransactionID', 'TransactionDT']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['isFraud']

X_val = val_df.drop(columns=drop_cols)
y_val = val_df['isFraud']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['isFraud']

print(f"X_train shape: {X_train.shape} | X_val shape: {X_val.shape} | X_test shape: {X_test.shape}")

### 2. Run Baseline and ML Model Training

Instead of running training inside the notebook, we import or trigger our reproducible `src/train_models.py` script to generate models and output results.

In [ ]:
# Run train_models.py script programmatically
!python ../src/train_models.py

### 3. Compare Models on Validation Set

We read and display `reports/model_comparison.csv` to see how the baseline Logistic Regression, LightGBM, and XGBoost models compare on PR-AUC, ROC-AUC, Precision, Recall, and F1.

In [ ]:
comparison_path = "../reports/model_comparison.csv"
comparison_df = pd.read_csv(comparison_path)
comparison_df

### 4. Threshold Optimization & Cost Minimization

Instead of standard 0.50, we run our threshold optimization script `src/evaluate_models.py` which computes precision, recall, and financial costs across thresholds ranging from 0.05 to 0.90.

In [ ]:
# Run evaluate_models.py script programmatically to generate plots, SHAP summary, and test set results
!python ../src/evaluate_models.py

#### Display Threshold Metrics
We review the threshold analysis and locate the threshold that minimizes Expected Financial Cost.

In [ ]:
threshold_df = pd.read_csv("../reports/threshold_analysis.csv")
threshold_df

#### Optimal Threshold Business Impact

We plot how the expected financial cost varies with the threshold. 

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(threshold_df['Threshold'], threshold_df['ExpectedCost'], marker='o', color='crimson', label='Expected Financial Cost')
plt.axvline(x=0.10, color='darkblue', linestyle='--', label='Optimal Threshold (0.10)')
plt.xlabel('Classification Threshold')
plt.ylabel('Expected Financial Cost ($)')
plt.title('Business Expected Cost vs. Classification Threshold')
plt.legend()
plt.show()

### 5. View Evaluation Curves

We display the ROC, Precision-Recall, Confusion Matrix, and Calibration curves generated by the pipeline.

In [ ]:
from IPython.display import Image, display

print("Precision-Recall Curve:")
display(Image(filename="../plots/precision_recall_curve.png"))

print("Calibration Curve:")
display(Image(filename="../plots/calibration_curve.png"))

print("Confusion Matrix (Threshold = 0.10):")
display(Image(filename="../plots/confusion_matrix.png"))

### 6. Explainability (SHAP & Gini Feature Importance)

We load Gini importance and display the SHAP global summary plot.

In [ ]:
print("Top 20 Features (Gini Importance):")
importance_df = pd.read_csv("../reports/feature_importance.csv")
display(importance_df.head(20))

print("SHAP Summary Plot (Global Impact on Sample):")
display(Image(filename="../plots/shap_summary.png"))

#### Local Explanations JSON
We print the local SHAP risk factors calculated for sample transactions.

In [ ]:
with open("../reports/local_explanations.json", "r", encoding="utf-8") as f:
    local_exp = json.load(f)
    
for i, exp in enumerate(local_exp):
    actual = "Fraud" if exp['ActualLabel'] == 1 else "Legitimate"
    print(f"\n=== Transaction ID: {exp['TransactionIndex']} (Actual: {actual}) ===")
    print(f"Predicted Fraud Probability: {exp['FraudProbability']:.2%}")
    print("Top Risk Factors:")
    for factor in exp['TopPositiveFactors'][:3]:
        print(f"  - {factor}")
    print("Top Mitigating Factors:")
    for factor in exp['TopNegativeFactors'][:3]:
        print(f"  - {factor}")

### 7. Untouched Test Set Performance (Single Evaluation)

We load and print the final metrics evaluated exactly once on the untouched temporal test set using the operational threshold of 0.10.

In [ ]:
with open("../reports/test_metrics.json", "r", encoding="utf-8") as f:
    test_results = json.load(f)
    
print("==================================================")
print("FINAL UNTOUCHED TEST SET METRICS")
print("==================================================")
print(f"Model Selected: {test_results['Model']}")
print(f"Operating Threshold: {test_results['Threshold']}")
print(f"PR-AUC (Average Precision): {test_results['Metrics']['PR-AUC']:.4f}")
print(f"ROC-AUC: {test_results['Metrics']['ROC-AUC']:.4f}")
print(f"Precision: {test_results['Metrics']['Precision']:.4f}")
print(f"Recall: {test_results['Metrics']['Recall']:.4f}")
print(f"F1 Score: {test_results['Metrics']['F1']:.4f}")
print(f"False Positives: {test_results['Metrics']['FP']:,} | False Negatives: {test_results['Metrics']['FN']:,}")
print(f"True Positives: {test_results['Metrics']['TP']:,} | True Negatives: {test_results['Metrics']['TN']:,}")
print(f"Expected Cost at optimal threshold: ${test_results['ExpectedCost']:,.2f}")
print(f"Baseline cost without fraud management: ${test_results['BaselineCost']:,.2f}")
print(f"Net Financial Benefit: ${test_results['NetBenefit']:,.2f}")
print("==================================================")